# Initial Train Dataset Build (Ingestion-Only)
Runs: WHO API probe -> X load -> Y fetch -> Polars preprocessing.

In [1]:
from pathlib import Path
import subprocess
import polars as pl

In [2]:
repo = Path.cwd().resolve()
if not (repo / 'ml' / 'scripts').exists():
    repo = repo.parent.parent
steps = [
    ['python', 'ml/scripts/probe_who_news_apis.py', '--out', 'ml/data/raw/who_api_probe.json'],
    ['python', 'ml/scripts/load_x_from_db.py', '--db-path', 'backend/app.db', '--out', 'ml/data/processed/x_candidates.parquet'],
    ['python', 'ml/scripts/fetch_y_who_news.py', '--out-json', 'ml/data/raw/y_who_news_raw.json', '--out-parquet', 'ml/data/processed/y_candidates.parquet', '--top', '100'],
    ['python', 'ml/scripts/preprocess_xy_polars.py', '--x', 'ml/data/processed/x_candidates.parquet', '--y', 'ml/data/processed/y_candidates.parquet', '--out', 'ml/data/processed/ml_ready.parquet'],
]
for cmd in steps:
    print('running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=repo, check=True)


running: python ml/scripts/probe_who_news_apis.py --out ml/data/raw/who_api_probe.json
endpoint=https://www.who.int/api/news/dons count=0 publication_min=None publication_max=None
endpoint=https://www.who.int/api/news/emergencies count=34 publication_min=2021-05-17T15:15:43+00:00 publication_max=2024-10-09T10:12:17+00:00
endpoint=https://www.who.int/api/news/outbreaks count=0 publication_min=None publication_max=None
running: python ml/scripts/load_x_from_db.py --db-path backend/app.db --out ml/data/processed/x_candidates.parquet
wrote 0 rows to ml/data/processed/x_candidates.parquet
running: python ml/scripts/fetch_y_who_news.py --out-json ml/data/raw/y_who_news_raw.json --out-parquet ml/data/processed/y_candidates.parquet --top 100
source=emergencies rows=34 out=ml/data/processed/y_candidates.parquet
running: python ml/scripts/preprocess_xy_polars.py --x ml/data/processed/x_candidates.parquet --y ml/data/processed/y_candidates.parquet --out ml/data/processed/ml_ready.parquet
wrote 34

In [3]:
ml_ready_path = repo / 'ml' / 'data' / 'processed' / 'ml_ready.parquet'
df = pl.read_parquet(ml_ready_path)
print('rows', df.height)
print('columns', df.columns)
print('ml_ready_path', ml_ready_path)
df.select(['y_source', 'publication_ts', 'risk_rating_norm', 'y_grade3_plus', 'y_event_start_t24h', 'y_event_start_t72h', 'x_value_mean', 'x_row_count']).head(10)


rows 34
columns ['y_source', 'record_id', 'title', 'publication_ts', 'new_outbreak', 'who_risk_assessment', 'emergency_start_ts', 'emergency_end_ts', 'summary', 'overview', 'raw_json', 'risk_rating_norm', 'y_grade3_plus', 'y_event_start_t24h', 'y_event_start_t72h', 'x_value_mean', 'x_value_std', 'x_row_count', 'x_indicator_nunique']
ml_ready_path /Users/yao/projects/biohack-2026-april/ml/data/processed/ml_ready.parquet


y_source,publication_ts,risk_rating_norm,y_grade3_plus,y_event_start_t24h,y_event_start_t72h,x_value_mean,x_row_count
str,"datetime[μs, UTC]",str,i64,i64,i64,f64,i64
"""emergencies""",2021-05-17 15:15:43 UTC,"""unknown""",0,0,0,null,null
"""emergencies""",2021-05-19 09:22:33 UTC,"""unknown""",0,0,0,null,null
"""emergencies""",2021-05-19 09:25:45 UTC,"""unknown""",0,0,0,null,null
"""emergencies""",2021-05-19 09:28:09 UTC,"""unknown""",0,0,0,null,null
"""emergencies""",2021-05-19 09:42:18 UTC,"""unknown""",0,0,0,null,null
"""emergencies""",2021-05-19 09:48:43 UTC,"""unknown""",0,0,0,null,null
"""emergencies""",2021-05-19 09:52:38 UTC,"""unknown""",0,0,0,null,null
"""emergencies""",2021-05-19 09:54:06 UTC,"""unknown""",0,0,0,null,null
"""emergencies""",2021-05-19 09:56:10 UTC,"""unknown""",0,0,0,null,null


In [4]:
from ydata_profiling import ProfileReport
report_path = repo / 'ml' / 'data' / 'processed' / 'ml_ready_profile.html'
pdf = df.to_pandas()
profile = ProfileReport(pdf, title='ML Ready Dataset Profile', minimal=True)
report_path.parent.mkdir(parents=True, exist_ok=True)
profile.to_file(report_path)
print('profile_report', report_path)


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 19/19 [00:00<00:00, 275750.09it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

profile_report /Users/yao/projects/biohack-2026-april/ml/data/processed/ml_ready_profile.html
